# Data Ingestion 

## 1. Create Widget

In [0]:
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp

# Start Time
start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

run_id = f"RUN_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Widgets

dbutils.widgets.text("table_metadata", "")
dbutils.widgets.text("table_parameters", "")
dbutils.widgets.text("run_id", "")

run_id = dbutils.widgets.get("run_id")
raw_metadata = dbutils.widgets.get("table_metadata")
raw_parameters = dbutils.widgets.get("table_parameters")

table_metadata = json.loads(raw_metadata)
table_parameters = json.loads(raw_parameters)

if isinstance(table_metadata, list):
    table_metadata = table_metadata[0]

print("Table Metadata:")
print(json.dumps(table_metadata, indent=4))

print("\nTable Parameters:")
print(json.dumps(table_parameters, indent=4))

print(f"\nRun ID: {run_id}")

## 2. Extract Variables

In [0]:
table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
source_system = table_metadata["source_system"].lower()
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]
bronze_schema = table_metadata["bronze_schema"]

load_type = table_parameters.get("load_type")
watermark_column = table_parameters.get("watermark_column")

bronze_table_fqn = f"banking.{bronze_schema}.{table_name}"

print(f"Target Bronze Table: {bronze_table_fqn}")

## 3. Insert or update audit table

In [0]:
entry_exists = spark.sql(f"""
    SELECT 1
    FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id} AND table_id = {table_id}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            layer = 'Silver',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id} AND table_id = {table_id}
    """)
else:
    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Silver',
            TIMESTAMP('{start_time}'),
            NULL,  -- end time
            'INPROGRESS',
            NULL, --number of records
            NULL -- error message
        )
    """)

## 4. Get Last Watermark (For Filtering Only)

In [0]:
last_watermark = None

if load_type in ["APPEND", "MERGE"] and watermark_column:
    watermark_df = spark.sql(f"""
        SELECT last_watermark_value
        FROM banking.metadata.table_watermarks
        WHERE table_id = {table_id}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["last_watermark_value"]

print("Last Watermark:", last_watermark)

In [0]:
%sql
create schema if not exists banking.bronze

## 5. Read Source
Auto Loader streaming ingestion from a Databricks Volume for Bronze layer.
Automatically detects new CSV files, handles schema inference/evolution, and ensures reliable processing using schemaLocation for metadata tracking.

In [0]:

try:
    source_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"/Volumes/banking/source/volume/_schema/{table_name}")
        .option("header", "true")
        .load("/Volumes/banking/source/volume/")
    )   

except Exception as e:

    print(f"Error reading CSV file: {e}")
    raise

# Add insert_timestamp

source_df = source_df.withColumn(
    "insert_timestamp",
    current_timestamp()
)

## 6. Write to Bronze (Append Only)

In [0]:
try:

    source_df = (
        spark.read
        .option("header", True)
        .csv(f"/Volumes/banking/source/volume/{table_name}.csv")
    )

    if not spark.catalog.tableExists(bronze_table_fqn):

        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(bronze_table_fqn)
        )

    else:

        (
            source_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(bronze_table_fqn)
        )

    print("Bronze load successful")

except Exception as e:

    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET end_time = TIMESTAMP('{end_time}'),
            status = 'FAILED',
            error_message = '{str(e).replace("'", "")}'
        WHERE table_id = {table_id}
        AND run_id = '{run_id}'
    """)

    raise